# Pulse sequence visualizer

Point `DATA_FOLDER` at any folder produced by `rt.deploy(...)` and run the notebook.
No hardware needed — the runtime is re-executed off-hardware up to its first
`acadia.run()` and the compiled schedule is rendered.

See `../README.md` for how it works and what the marks mean.

In [ ]:
import logging
import sys
from pathlib import Path

# make `sequence_viz` importable regardless of where the notebook is opened from
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "sequence_viz" / "__init__.py").is_file():
        sys.path.insert(0, str(candidate))
        break

logging.getLogger("acadia").setLevel(logging.ERROR)  # acadia is chatty at DEBUG

import matplotlib.pyplot as plt
import sequence_viz as sv

%matplotlib inline

## 1. Choose a data folder

In [ ]:
DATA_FOLDER = "/path/to/data_folder"

POINT = 0                    # which acadia.run() to stop at (sweep point index)
RESOLVE_INDETERMINATE = 0    # cycles to assume for register-driven lengths

assert sv.is_data_folder(DATA_FOLDER), f"not a deployed data folder: {DATA_FOLDER}"
sorted(p.name for p in Path(DATA_FOLDER).iterdir())

## 2. Trace it

In [ ]:
trace = sv.trace_folder(
    DATA_FOLDER,
    point=POINT,
    resolve_indeterminate=RESOLVE_INDETERMINATE,
    # use_saved_qmsmt=False,        # trace against the installed acadia_qmsmt instead
    # overrides={"iterations": 1},  # shrink a field before the dry run if needed
)
print(trace.summary())

## 3. Full sequence

In [ ]:
fig, ax = sv.plot_trace(trace)
plt.show()

## 4. Zoom in

Envelopes, barriers and alignment padding only become readable once you zoom.
`xlim_ns` is in nanoseconds regardless of the axis unit shown.

In [ ]:
tail_ns = trace.length_ns
fig, ax = sv.plot_trace(
    trace,
    xlim_ns=(max(0, tail_ns - 3500), tail_ns + 500),
    title=f"{trace.runtime_class} — readout region",
)
plt.show()

## 5. Textual timeline

Exact numbers per command. `pad` marks a dwell acadia inserted to align channels
at a barrier; `user` is something the runtime scheduled.

In [ ]:
print(trace.to_text(max_blocks=6))

## 6. Check the trace against what actually ran

`compiled.log` in the folder is the sequencer program the FPGA executed that day.
`match: True` means the re-trace reproduces it exactly.

In [ ]:
check = sv.compare_with_compiled_log(trace, DATA_FOLDER)
for key, value in check.items():
    print(f"{key:20s} {value}")

## 7. A live runtime instead of a folder

Same picture before deploying — build the runtime as usual and pass the object.

In [ ]:
# from acadia_qmsmt.runtimes.one_qubit_tomography import OneQubitTomographyRuntime
# rt = SomeRuntime(**config)
# fig, ax, trace = sv.plot_runtime(rt)
# plt.show()